# Session 9: AI Agents — Concepts and Patterns

This notebook covers:
1. **Agentic RAG** — the bridge from Session 8
2. **LLM vs. Agent** — what changes when you add a control loop
3. **Workflow patterns** — prompt chaining, routing, parallelization, evaluator-optimizer
4. **MCP** — the universal wiring layer for agent tools

We build a small FAISS index inline so everything is self-contained.

In [ ]:
import os
import sys
import numpy as np
import faiss
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("Setup complete")

## 1. Agentic RAG — The Bridge from Session 8

In Session 8 we built **single-shot RAG**:
```
query → embed → retrieve → generate → done
```

**Agentic RAG** adds a loop around that pipeline. But there are two very different ways
to implement that loop — and the difference is the central lesson of this session.

**Version 1 (this section):** The programmer writes the loop. We decide when to check
completeness, when to rewrite the query, when to stop. The LLM answers questions,
but the control flow is entirely ours.

**Version 2 (notebook 02):** We give the LLM a `retrieve` tool and let it run its own loop.
The LLM decides when to search, what to search for, and when it has enough to answer.

Version 1 is a **workflow**. Version 2 is an **agent**.
We build Version 1 first so the difference is concrete when we get to Version 2.

We start by building a small in-memory FAISS index over 10 sample texts.

In [ ]:
SAMPLE_TEXTS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.",
    "FAISS is a library for efficient similarity search over dense vectors, built by Meta.",
    "Cosine similarity measures the angle between two vectors, returning a value between -1 and 1.",
    "An embedding is a dense numerical representation of text that captures semantic meaning.",
    "BM25 is a lexical search algorithm scoring by term frequency and inverse document frequency.",
    "The OpenAI Agents SDK uses Agent, Runner, function_tool, and handoffs as its core primitives.",
    "Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.",
    "MCP (Model Context Protocol) is an open standard for connecting AI models to tools and data sources.",
    "Prompt chaining passes the output of one LLM call as the input to the next in a pipeline.",
    "The compound accuracy problem: a 10-step workflow at 85% accuracy per step succeeds only 20% of the time.",
]

def embed_texts(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in response.data], dtype=np.float32)

def embed_query(query: str) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=[query])
    return np.array(response.data[0].embedding, dtype=np.float32)

embeddings = embed_texts(SAMPLE_TEXTS)
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
print(f"FAISS index ready: {faiss_index.ntotal} vectors, dimension {embeddings.shape[1]}")

### The Agentic RAG Loop

Four helper functions, each doing one job:
- `retrieve()` — embed the query and find nearest chunks in FAISS
- `generate()` — call the LLM with retrieved context
- `is_complete()` — ask the LLM if the answer is sufficient
- `rewrite_query()` — ask the LLM what to search for next

In [ ]:
def retrieve(query: str, top_k: int = 2) -> list[str]:
    """Find the top-k most relevant chunks for a query."""
    q_vec = embed_query(query).reshape(1, -1)
    _, indices = faiss_index.search(q_vec, top_k)
    return [SAMPLE_TEXTS[i] for i in indices[0]]

def generate(question: str, context_chunks: list[str]) -> str:
    """Generate an answer grounded in the retrieved context."""
    context = "\n\n".join(context_chunks)
    return client.responses.create(
        model="gpt-4o-mini",
        instructions=(
            "Answer using only the provided context. "
            "If context is insufficient, say exactly what information is missing."
        ),
        input=f"Context:\n{context}\n\nQuestion: {question}"
    ).output_text

def is_complete(question: str, answer: str) -> bool:
    """Ask the LLM whether the answer fully addresses the original question."""
    reply = client.responses.create(
        model="gpt-4o-mini",
        instructions="Reply only 'yes' or 'no'. Does this answer fully address the question?",
        input=f"Question: {question}\n\nAnswer: {answer}"
    ).output_text.strip().lower()
    return reply.startswith("yes")

def rewrite_query(original_question: str, partial_answer: str) -> str:
    """Generate a better search query to find what's still missing."""
    # Simplification: always rewrites from the original question + current partial answer.
    # A production loop would also pass the current query to avoid regenerating the same rewrite.
    return client.responses.create(
        model="gpt-4o-mini",
        instructions="Write a short search query (under 10 words) to find the missing information.",
        input=f"Original question: {original_question}\nPartial answer: {partial_answer}"
    ).output_text.strip()

In [ ]:
def agentic_rag(question: str, max_iterations: int = 3) -> str:
    """
    Agentic RAG control loop:
      1. Retrieve relevant chunks for the current query
      2. Generate an answer from accumulated context
      3. If complete → return. If not → rewrite query and loop.
    """
    query = question
    accumulated_context: list[str] = []

    print(f"Question: {question}\n{'='*60}")

    for i in range(max_iterations):
        print(f"\n[Iteration {i+1}] Searching for: {query}")
        new_chunks = [c for c in retrieve(query) if c not in accumulated_context]
        accumulated_context.extend(new_chunks)
        print(f"  {len(new_chunks)} new chunk(s) retrieved (total: {len(accumulated_context)})")

        answer = generate(question, accumulated_context)
        preview = answer[:120] + ("..." if len(answer) > 120 else "")
        print(f"  Answer: {preview}")

        if is_complete(question, answer):
            print(f"\n✓ Complete after {i+1} iteration(s)")
            return answer

        query = rewrite_query(question, answer)
        print(f"  → Rewriting query: {query}")

    print(f"\n⚠ Reached max iterations ({max_iterations})")
    return answer

answer = agentic_rag(
    "How does FAISS support semantic search, and what similarity metric does it use?"
)
print(f"\nFinal answer:\n{answer}")

### What the programmer still controls

Look at `agentic_rag()` and ask: *who is making each decision?*

| Decision | Who makes it? |
|---|---|
| How many times to try | Programmer (`max_iterations = 3`) |
| Whether to check completeness | Programmer (calls `is_complete()` every iteration) |
| Whether to rewrite the query | Programmer (always calls `rewrite_query()` if not complete) |
| When to stop | Programmer (first `is_complete()` → True, or max reached) |

The LLM is answering questions inside each step — but the **control flow is entirely ours**.
This is a workflow that *simulates* agentic behaviour, not a true agent.

**A true agent:** give the LLM a `retrieve` tool and let `Runner` manage the loop.
The LLM then decides *if* to retrieve, *what* to search for, and *when* it has enough.
We build that in notebook 02.

## 2. LLM vs. Agent — What Changes When You Add a Loop

| | LLM | Agent |
|---|---|---|
| Autonomy | Reacts to prompts | Acts toward a goal |
| Tool use | Needs explicit calling | Decides when to call tools |
| Memory | Stateless | Accumulates context |
| Workflow | Single-step | Multi-step, iterative |

The core difference: **an agent decides its own next action**.
A static LLM call is determined entirely by the caller.

In [ ]:
QUESTION = (
    "What is cosine similarity and what is 0.85 raised to the power of 10? "
    "Also, what does FAISS stand for?"
)

# Static LLM: one call, no tools, relies entirely on training data
print("=== Static LLM (one call, no tools) ===")
static_answer = client.responses.create(
    model="gpt-4o-mini",
    instructions="Answer the question concisely.",
    input=QUESTION
).output_text
print(static_answer)

print("\n" + "="*60)
print("=== What an agent would do differently ===")
print("""
1. Recognise this question has three independent sub-tasks
2. Call a search tool for 'cosine similarity' → retrieve definition
3. Call a calculator tool for 0.85 ** 10 → exact result
4. Call a search tool for 'FAISS' → retrieve acronym
5. Combine results into one coherent answer

The agent decides WHEN and WHICH tools to call.
We build this in notebook 02 using the OpenAI Agents SDK.
""")

## 3. Workflow Patterns

Six patterns from Anthropic's "Building Effective Agents" article.
We implement four as working code examples.

### Pattern 1: Prompt Chaining

Output of one LLM call feeds the next. Good for tasks with clear sequential stages.

```
raw input → [extract key points] → [formalise] → [headline]
```

In [ ]:
def prompt_chain(raw_notes: str) -> dict:
    """Three-step chain: extract → formalise → headline."""
    print("Step 1: Extract key points")
    key_points = client.responses.create(
        model="gpt-4o-mini",
        instructions="Extract exactly 3 key points as a bullet list.",
        input=raw_notes
    ).output_text
    print(key_points)

    print("Step 2: Rewrite in formal language")
    formal = client.responses.create(
        model="gpt-4o-mini",
        instructions="Rewrite these bullet points in formal business language.",
        input=key_points
    ).output_text
    print(formal)

    print("Step 3: Generate a one-sentence headline")
    headline = client.responses.create(
        model="gpt-4o-mini",
        instructions="Write a single-sentence executive headline summarising these points.",
        input=formal
    ).output_text
    print(f"\n📰 Headline: {headline}")

    return {"key_points": key_points, "formal": formal, "headline": headline}

RAW_NOTES = """
Session 9 covered AI agents. We learned that agents use a control loop to decide
when and how to use tools. We also covered MCP, which standardises how agents
connect to external tools. The hands-on session used the OpenAI Agents SDK to
build a tool-using agent with FAISS-based search.
"""

chain_result = prompt_chain(RAW_NOTES)

### Pattern 2: Routing

Classify the input first, then send it to the right handler.
Used in customer service, triage systems, and intent-based pipelines.

In [ ]:
HANDLERS = {
    "billing":   lambda q: f"[Billing Agent] Processing payment query: {q}",
    "technical": lambda q: f"[Tech Support] Troubleshooting: {q}",
    "returns":   lambda q: f"[Returns Agent] Processing return request: {q}",
    "other":     lambda q: f"[General Agent] Answering: {q}",
}

def route_and_handle(query: str) -> str:
    intent = client.responses.create(
        model="gpt-4o-mini",
        instructions=(
            "Classify this customer query as exactly one of: "
            "billing, technical, returns, other. Reply with one word only."
        ),
        input=query
    ).output_text.strip().lower()

    category = intent if intent in HANDLERS else "other"
    response = HANDLERS[category](query)
    print(f"Query:    {query}")
    print(f"Routed to: {category}")
    print(f"Response: {response}\n")
    return response

TEST_QUERIES = [
    "My invoice shows double the amount I should be paying",
    "The app keeps crashing when I open the settings page",
    "I want to return the headphones I bought last week",
    "What time does your support team work?",
]

for q in TEST_QUERIES:
    route_and_handle(q)

### Pattern 3: Parallelization

Run independent LLM calls concurrently using `ThreadPoolExecutor`.
Reduces total latency when sub-tasks don't depend on each other.

*Note: these are three separate API calls running at the same time.*

In [ ]:
PAPERS = [
    {
        "title": "RAG (Lewis et al., 2020)",
        "text": (
            "Retrieval-Augmented Generation combines a dense retriever with a seq2seq generator "
            "to ground answers in retrieved documents, reducing hallucination on open-domain QA."
        ),
    },
    {
        "title": "Sentence-BERT (Reimers & Gurevych, 2019)",
        "text": (
            "Sentence-BERT uses siamese BERT networks with mean pooling to produce fixed-length "
            "sentence embeddings suited for semantic similarity via cosine distance."
        ),
    },
    {
        "title": "FAISS (Johnson et al., 2019)",
        "text": (
            "FAISS provides efficient exact and approximate nearest-neighbor search over "
            "billion-scale dense vectors using GPU-accelerated IVF and HNSW index types."
        ),
    },
]

def summarise_one(paper: dict) -> tuple[str, str]:
    summary = client.responses.create(
        model="gpt-4o-mini",
        instructions="Summarise in exactly one sentence for a Master's student.",
        input=paper["text"]
    ).output_text.strip()
    return paper["title"], summary

print("Summarising 3 papers in parallel...\n")
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(summarise_one, PAPERS))

for title, summary in results:
    print(f"📄 {title}")
    print(f"   {summary}\n")

### Pattern 4: Evaluator-Optimizer

One agent generates output. Another scores it and gives feedback.
The generator improves until the score is high enough.

Used in: content pipelines, coding agent loops, automated essay writing.

In [ ]:
def evaluator_optimizer(task: str, target_score: int = 8, max_rounds: int = 3) -> str:
    """Generate → evaluate → improve loop. Stop when score ≥ target_score."""
    print(f"Task: {task}\n{'='*60}")

    draft = client.responses.create(
        model="gpt-4o-mini",
        instructions="Complete the task to the best of your ability.",
        input=task
    ).output_text

    for round_num in range(1, max_rounds + 1):
        print(f"\n[Round {round_num}] Draft preview:")
        print(draft[:180] + ("..." if len(draft) > 180 else ""))

        evaluation = client.responses.create(
            model="gpt-4o-mini",
            instructions=(
                "Evaluate this output on a scale of 1–10. "
                "Reply in exactly this format, nothing else:\n"
                "SCORE: <number>\n"
                "FEEDBACK: <one sentence of specific improvement needed>"
            ),
            input=f"Task: {task}\n\nOutput:\n{draft}"
        ).output_text

        lines = {
            line.split(":", 1)[0].strip(): line.split(":", 1)[1].strip()
            for line in evaluation.splitlines()
            if ":" in line
        }
        score = int(lines.get("SCORE", "0").split()[0])
        feedback = lines.get("FEEDBACK", "No specific feedback")

        print(f"Score: {score}/10 | Feedback: {feedback}")

        if score >= target_score:
            print(f"\n✓ Accepted at round {round_num} with score {score}/10")
            return draft

        draft = client.responses.create(
            model="gpt-4o-mini",
            instructions="Improve the output based on the feedback. Keep the same format.",
            input=f"Task: {task}\n\nCurrent output:\n{draft}\n\nFeedback: {feedback}"
        ).output_text

    print(f"\n⚠ Reached max rounds. Returning best draft (score {score}/10)")
    return draft

final = evaluator_optimizer(
    "Explain what a vector embedding is to a first-year data science student. Use one concrete analogy."
)
print(f"\nFinal output:\n{final}")

### Aside: where structured output would improve this

The evaluator currently returns free text in a format we parse manually:
```
SCORE: 7
FEEDBACK: Add a concrete analogy
```

This is fragile — the model might vary the format and our `split(":")` breaks.
The robust version uses a Pydantic model as the response schema:

```python
class Evaluation(BaseModel):
    score: int        # 1–10
    feedback: str     # one sentence of specific improvement

# Then parse with:
evaluation = client.responses.parse(
    model="gpt-4o-mini",
    text={"format": {"type": "json_schema", "schema": Evaluation.model_json_schema(), "strict": True}},
    input=f"Evaluate: {draft}"
)
score = evaluation.score     # int, guaranteed
feedback = evaluation.feedback  # str, guaranteed
```

We keep the string-parsing version here to keep the pattern readable.
In production, always use structured output for anything you need to act on programmatically.

## 4. Model Context Protocol (MCP)

### The Problem: M × N Integrations

Before MCP, every AI application needed custom code to talk to every tool:
- Claude talking to GitHub = custom integration
- Claude talking to Notion = another custom integration
- ChatGPT talking to GitHub = another custom integration
- ...M applications × N tools = M×N bespoke integrations

### The Solution: M + N

MCP defines a **single standard interface** (JSON-RPC based):
- Each AI **client** (Claude, ChatGPT, Cursor, VS Code) implements MCP once
- Each **tool server** (GitHub, Notion, Postgres, your own app) implements MCP once
- They connect without any custom glue code

Analogy: **"USB-C for language models"** — one standard plug for any context source.

### Adoption (April 2026)
- Introduced by Anthropic, November 2024
- Donated to Linux Foundation (Agentic AI Foundation), April 2026
- 97 million monthly SDK downloads
- 10,000+ public MCP servers
- Supported by: Claude, ChatGPT, Cursor, VS Code, GitHub Copilot, AWS, Azure, GCP

In [ ]:
print("""
BEFORE MCP: M apps × N tools = M×N integrations
─────────────────────────────────────────────────────
  Claude ──── custom ──── GitHub API
  Claude ──── custom ──── Notion API
  Claude ──── custom ──── Postgres
  ChatGPT ─── custom ──── GitHub API
  ChatGPT ─── custom ──── Notion API
  Cursor ──── custom ──── GitHub API
  (every pair needs its own code)

AFTER MCP: M+N — each side implements once
─────────────────────────────────────────────────────
  Claude  ──────┐
  ChatGPT ──────┼── MCP Client ══ MCP Protocol ══ GitHub MCP Server
  Cursor  ──────┘                               ╠══ Notion MCP Server
                                                ╚══ Your App MCP Server
""")

### Building an MCP Server

The `mcp` Python package lets you define a server in a few lines.
Once running, any MCP-compatible client can call your tools automatically —
no additional integration code needed.

In [ ]:
from mcp.server.fastmcp import FastMCP

# Define an MCP server
mcp_server = FastMCP("ANLP Course Server")

@mcp_server.tool()
def get_session_topics(session_number: int) -> str:
    """Get the topics covered in a given ANLP course session."""
    topics = {
        7: "LLM Deep Dive, Prompting Techniques (CoT, few-shot), LLM Evaluation (BLEU, ROUGE, LLM-as-judge)",
        8: "LLM API Access, RAG, FAISS, Streaming, Gradio/Streamlit, Tool Use",
        9: "AI Agents, Workflow Patterns, MCP, OpenAI Agents SDK",
    }
    return topics.get(session_number, f"Session {session_number} not in course")

@mcp_server.tool()
def search_glossary(term: str) -> str:
    """Look up a term in the ANLP course glossary."""
    glossary = {
        "rag": "Retrieval-Augmented Generation — grounding LLM answers in retrieved documents",
        "embedding": "A dense numerical vector representing text meaning",
        "faiss": "Facebook AI Similarity Search — fast vector similarity search library",
        "mcp": "Model Context Protocol — open standard for connecting AI models to tools",
        "agent": "An LLM extended with tools that can take multi-step actions toward a goal",
        "bm25": "A lexical search algorithm scoring by term frequency and document rarity",
        "cosine similarity": "A measure of angle between two vectors, bounded between -1 and 1",
    }
    return glossary.get(term.lower(), f"'{term}' not in glossary — try a related term")

# Test the tools directly (they are just Python functions)
print(get_session_topics(8))
print()
print(search_glossary("mcp"))

### How to run this as a real MCP server

Add `mcp_server.run()` at the end of a Python file and run it.
Then add it to any MCP client's config. Example for **Claude Desktop**:

```json
{
  "mcpServers": {
    "anlp": {
      "command": "python",
      "args": ["path/to/your_server.py"]
    }
  }
}
```

Once connected, Claude Desktop can call `get_session_topics()` and
`search_glossary()` from any conversation — with no extra code on Claude's side.

The same server also works with **Cursor**, **VS Code**, and **GitHub Copilot**
without changing a single line of the server implementation.

## Summary

| Concept | What it does |
|---|---|
| **Agentic RAG** | LLM controls its own retrieval loop — retrieves, checks, rewrites, retrieves again |
| **Prompt chaining** | Sequential LLM calls; output of each step is input to the next |
| **Routing** | Classify input → send to the right specialist |
| **Parallelization** | Run independent LLM calls concurrently |
| **Evaluator-optimizer** | Generate → score → improve until quality threshold is met |
| **MCP** | Standard protocol so any agent can call any tool with one implementation |

**Next: notebook 02** — build real agents using the OpenAI Agents SDK.